# Sentiment Classification using Spacy

In [1]:
import numpy as np
import pandas as pd
import re
import string


# Load the data

In [2]:
df = pd.read_csv("./../data/tweet_sentiment.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   tweet      1000 non-null   str  
 1   sentiment  1000 non-null   str  
dtypes: str(2)
memory usage: 15.8 KB


In [3]:
df.head()

,tweet,sentiment
0,The event starts at 5 PM.,neutral
1,I hate how this turned out.,negative
2,Fantastic experience!,positive
3,Fantastic experience!,positive
4,This is the worst thing ever!,negative


In [4]:
df.value_counts("sentiment")

sentiment
positive    396
neutral     317
negative    287
Name: count, dtype: int64

In [5]:
df['label'] = df['sentiment'].map({'negative': 0, 'positive': 1, 'neutral': 2})

print(df.head())

df['label'].value_counts()

                           tweet sentiment  label
0      The event starts at 5 PM.   neutral      2
1    I hate how this turned out.  negative      0
2          Fantastic experience!  positive      1
3          Fantastic experience!  positive      1
4  This is the worst thing ever!  negative      0


label
1    396
2    317
0    287
Name: count, dtype: int64

# Preprocessing

In [6]:
def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags (iOS)
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )
    return emoji_pattern.sub(r"", text)

In [7]:
def remove_url(text):
    url_pattern = re.compile(r"http\S+|www\S+|https\S+", re.IGNORECASE)
    return url_pattern.sub(r"", text)

In [8]:
def clean_text(text):
    delete_dict = {sp_character: "" for sp_character in string.punctuation}
    delete_dict[" "] = " "
    table = str.maketrans(delete_dict)
    text1 = text.translate(table)
    textArr = text1.split()
    return " ".join(
        [word for word in textArr if (not word.isdigit() and (
            not word.isdigit() and len(word) > 3
        ))]
    ).lower()

In [9]:
df['processed_tweet'] = df['tweet'].apply(remove_emoji).apply(remove_url).apply(clean_text)
df.head()

,tweet,sentiment,label,processed_tweet
0,The event starts at 5 PM.,neutral,2,event starts
1,I hate how this turned out.,negative,0,hate this turned
2,Fantastic experience!,positive,1,fantastic experience
3,Fantastic experience!,positive,1,fantastic experience
4,This is the worst thing ever!,negative,0,this worst thing ever


In [10]:
# One hot encoding the labels
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df[['sentiment']])

encoded_df = pd.DataFrame(
    encoded, 
    columns=encoder.get_feature_names_out(['sentiment']),
    index=df.index
)

df = pd.concat([df, encoded_df], axis=1)

df.head()

,tweet,sentiment,label,processed_tweet,sentiment_negative,sentiment_neutral,sentiment_positive
0,The event starts at 5 PM.,neutral,2,event starts,0.0,1.0,0.0
1,I hate how this turned out.,negative,0,hate this turned,1.0,0.0,0.0
2,Fantastic experience!,positive,1,fantastic experience,0.0,0.0,1.0
3,Fantastic experience!,positive,1,fantastic experience,0.0,0.0,1.0
4,This is the worst thing ever!,negative,0,this worst thing ever,1.0,0.0,0.0


In [11]:
# Sanity check
df.filter(like="sentiment_").sum(), df['sentiment'].value_counts()

(sentiment_negative    287.0
 sentiment_neutral     317.0
 sentiment_positive    396.0
 dtype: float64,
 sentiment
 positive    396
 neutral     317
 negative    287
 Name: count, dtype: int64)

# Use Spacy to process further.

### Install dependencies and import for transformers

The next cells ensure `spacy-transformers` is available and bring in training utilities.

In [12]:
# # install only if not already present
# !pip install -U spacy-transformers
# # download model if missing
# !python -m spacy download en_core_web_trf

In [13]:
import spacy
from spacy.util import minibatch, compounding
import random

In [14]:
# Create a simple blank model with textcat (no transformers for simpler debugging)
nlp = spacy.blank("en")

# Create textcat component with default architecture (BOW)
config = {
    "model": {
        "@architectures": "spacy.TextCatBOW.v3",
        "exclusive_classes": True,
        "length": 262144,
        "ngram_size": 1,
        "no_output_layer": False,
    },
    "threshold": 0.5,
}
textcat = nlp.add_pipe("textcat", config=config)

for label in ["negative", "positive", "neutral"]:
    textcat.add_label(label)

print(f"Pipeline: {nlp.pipe_names}")

e:\Projects\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pipeline: ['textcat']


### Prepare training data
Convert cleaned tweets into spaCy's `(text, {"cats": ...})` format and split.

In [15]:
from spacy.training import Example

def make_cats(label):
    return {"negative": label == 0, "positive": label == 1, "neutral": label == 2}

# Create Example objects for spaCy 3.x training
TRAIN_DATA = []
for text, lbl in zip(df.processed_tweet, df.label):
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, {"cats": make_cats(lbl)})
    TRAIN_DATA.append(example)

In [16]:
TRAIN_DATA[:5]

[{'doc_annotation': {'cats': {'negative': False, 'positive': False, 'neutral': True}, 'entities': ['O', 'O'], 'spans': {}, 'links': {}}, 'token_annotation': {'ORTH': ['event', 'starts'], 'SPACY': [True, False], 'TAG': ['', ''], 'LEMMA': ['', ''], 'POS': ['', ''], 'MORPH': ['', ''], 'HEAD': [0, 1], 'DEP': ['', ''], 'SENT_START': [1, 0]}},
 {'doc_annotation': {'cats': {'negative': True, 'positive': False, 'neutral': False}, 'entities': ['O', 'O', 'O'], 'spans': {}, 'links': {}}, 'token_annotation': {'ORTH': ['hate', 'this', 'turned'], 'SPACY': [True, True, False], 'TAG': ['', '', ''], 'LEMMA': ['', '', ''], 'POS': ['', '', ''], 'MORPH': ['', '', ''], 'HEAD': [0, 1, 2], 'DEP': ['', '', ''], 'SENT_START': [1, 0, 0]}},
 {'doc_annotation': {'cats': {'negative': False, 'positive': True, 'neutral': False}, 'entities': ['O', 'O'], 'spans': {}, 'links': {}}, 'token_annotation': {'ORTH': ['fantastic', 'experience'], 'SPACY': [True, False], 'TAG': ['', ''], 'LEMMA': ['', ''], 'POS': ['', ''], 'MOR

In [17]:
from sklearn.model_selection import train_test_split
indices = list(range(len(TRAIN_DATA)))
train_idx, dev_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=df.label)

train_data = [TRAIN_DATA[i] for i in train_idx]
dev_data = [TRAIN_DATA[i] for i in dev_idx]

print(f"Train: {len(train_data)}, Dev: {len(dev_data)}")

Train: 800, Dev: 200


### Add and configure text classification component
We attach a `textcat` pipeline to the transformer model and register labels.

In [18]:
print("before pipeline:", nlp.pipe_names)
if "textcat" not in nlp.pipe_names:
    textcat = nlp.add_pipe("textcat", last=True)
else:
    textcat = nlp.get_pipe("textcat")

for label in ["negative", "positive", "neutral"]:
    textcat.add_label(label)

print("after pipeline:", nlp.pipe_names)

before pipeline: ['textcat']
after pipeline: ['textcat']


### Train the model
Run several epochs; adjust `n_iter` and `drop` to tune.

In [31]:
# Initialize textcat with examples
textcat = nlp.get_pipe("textcat")
textcat.initialize(lambda: train_data[:50])

# Create optimizer
optimizer = nlp.create_optimizer()
n_iter = 100

print("Starting training...")
for epoch in range(n_iter):
    random.shuffle(train_data)
    losses = {}
    
    # Train in batches
    for i in range(0, len(train_data), 32):
        batch = train_data[i:i + 32]
        nlp.update(batch, sgd=optimizer, drop=0.5, losses=losses)
    
    loss = losses.get("textcat", 0)
    print(f"Epoch {epoch+1}/{n_iter} | Loss: {loss:.4f}")

Starting training...
Epoch 1/100 | Loss: 5.1439
Epoch 2/100 | Loss: 4.3222
Epoch 3/100 | Loss: 3.5656
Epoch 4/100 | Loss: 2.8789
Epoch 5/100 | Loss: 2.2842
Epoch 6/100 | Loss: 1.7813
Epoch 7/100 | Loss: 1.3677
Epoch 8/100 | Loss: 1.0393
Epoch 9/100 | Loss: 0.7810
Epoch 10/100 | Loss: 0.5802
Epoch 11/100 | Loss: 0.4247
Epoch 12/100 | Loss: 0.3124
Epoch 13/100 | Loss: 0.2390
Epoch 14/100 | Loss: 0.1912
Epoch 15/100 | Loss: 0.1584
Epoch 16/100 | Loss: 0.1341
Epoch 17/100 | Loss: 0.1156
Epoch 18/100 | Loss: 0.1012
Epoch 19/100 | Loss: 0.0896
Epoch 20/100 | Loss: 0.0800
Epoch 21/100 | Loss: 0.0721
Epoch 22/100 | Loss: 0.0654
Epoch 23/100 | Loss: 0.0596
Epoch 24/100 | Loss: 0.0547
Epoch 25/100 | Loss: 0.0504
Epoch 26/100 | Loss: 0.0466
Epoch 27/100 | Loss: 0.0433
Epoch 28/100 | Loss: 0.0403
Epoch 29/100 | Loss: 0.0377
Epoch 30/100 | Loss: 0.0353
Epoch 31/100 | Loss: 0.0331
Epoch 32/100 | Loss: 0.0312
Epoch 33/100 | Loss: 0.0294
Epoch 34/100 | Loss: 0.0278
Epoch 35/100 | Loss: 0.0263
Epoch 36

### Evaluate the model
Compute simple accuracy on the dev split.

In [32]:
def evaluate(examples):
    correct = 0
    for example in examples:
        doc = nlp(example.text)
        pred = max(doc.cats, key=doc.cats.get)
        true_cats = example.reference.cats
        true = max(true_cats, key=true_cats.get)
        if pred == true:
            correct += 1
    return correct / len(examples)

print("dev accuracy:", evaluate(dev_data))

dev accuracy: 1.0


### Save and use the trained pipeline
Store the model and test on example text.

In [33]:
import os
from pathlib import Path

model_path = "./../models/tweet_sentiment_trf"
Path(model_path).mkdir(parents=True, exist_ok=True)

nlp.to_disk(model_path)
print(f"Model saved to {model_path}")

# reload example
# nlp2 = spacy.load(model_path)
# print(nlp2("I love this!").cats)

Model saved to ./../models/tweet_sentiment_trf


### Load and Use the Trained Model for Predictions

Load the saved model and make predictions on new tweets.


In [34]:
# Load the saved model
loaded_nlp = spacy.load("./../models/tweet_sentiment_trf")

# Test on new sample tweets
test_texts = [
    "I love this product, it's amazing!",
    "This is terrible and disappointing",
    "The weather is nice today",
    "I am not happy with the service",
    "This is the best day ever!"
]

print("Sentiment Predictions:\n" + "="*50)
for text in test_texts:
    doc = loaded_nlp(text)
    predicted_sentiment = max(doc.cats, key=doc.cats.get)
    confidence = max(doc.cats.values())
    print(f"Text: {text}")
    print(f"Sentiment: {predicted_sentiment}")
    print(f"Confidence: {confidence:.4f}")
    print(f"All scores: {doc.cats}\n")


Sentiment Predictions:
Text: I love this product, it's amazing!
Sentiment: negative
Confidence: 0.3585
All scores: {'negative': 0.3585006296634674, 'positive': 0.35582154989242554, 'neutral': 0.28567779064178467}

Text: This is terrible and disappointing
Sentiment: negative
Confidence: 0.6893
All scores: {'negative': 0.6893059015274048, 'positive': 0.17084312438964844, 'neutral': 0.13985100388526917}

Text: The weather is nice today
Sentiment: neutral
Confidence: 0.8220
All scores: {'negative': 0.07749464362859726, 'positive': 0.10046017169952393, 'neutral': 0.8220452070236206}

Text: I am not happy with the service
Sentiment: positive
Confidence: 0.7648
All scores: {'negative': 0.11342629790306091, 'positive': 0.7647834420204163, 'neutral': 0.12179034948348999}

Text: This is the best day ever!
Sentiment: negative
Confidence: 0.6387
All scores: {'negative': 0.63873690366745, 'positive': 0.18025410175323486, 'neutral': 0.18100903928279877}



In [35]:
# Function to preprocess and predict sentiment
def predict_sentiment(text, nlp_model=loaded_nlp):
    """
    Preprocess raw text and predict sentiment
    
    Args:
        text: Raw tweet/text to classify
        nlp_model: spaCy model to use for prediction
    
    Returns:
        dict: Contains predicted sentiment, confidence, and all scores
    """
    # Apply same preprocessing as training data
    cleaned_text = text.apply(remove_emoji).apply(remove_url).apply(clean_text) if isinstance(text, pd.Series) else remove_emoji(remove_url(clean_text(text)))
    
    # Process with model
    doc = nlp_model(cleaned_text)
    
    # Get prediction
    predicted_sentiment = max(doc.cats, key=doc.cats.get)
    confidence = max(doc.cats.values())
    
    return {
        "original_text": text,
        "cleaned_text": cleaned_text,
        "sentiment": predicted_sentiment,
        "confidence": confidence,
        "scores": doc.cats
    }

# Test the function
test_tweet = "I absolutely love this movie! Best film ever!"
result = predict_sentiment(test_tweet)

print("Prediction Result:")
print(f"Original: {result['original_text']}")
print(f"Cleaned: {result['cleaned_text']}")
print(f"Sentiment: {result['sentiment']}")
print(f"Confidence: {result['confidence']:.4f}")
print(f"All Scores: {result['scores']}")


Prediction Result:
Original: I absolutely love this movie! Best film ever!
Cleaned: absolutely love this movie best film ever
Sentiment: positive
Confidence: 0.6375
All Scores: {'negative': 0.3365834057331085, 'positive': 0.6374884247779846, 'neutral': 0.02592816948890686}


### Batch Prediction on Multiple Tweets

Process multiple tweets and display results in a DataFrame.


In [36]:
# Batch prediction on sample tweets
sample_tweets = [
    "I love this product! Amazing quality!",
    "This is absolutely terrible and useless",
    "The weather today is neutral",
    "I hate waiting in long lines",
    "Best purchase I've made in years!",
    "Not bad, could be better though",
    "This is the worst experience ever",
    "Great service and friendly staff",
]

# Predict for all samples
predictions = []
for tweet in sample_tweets:
    # Apply preprocessing
    cleaned = remove_emoji(remove_url(clean_text(tweet)))
    
    # Get prediction
    doc = loaded_nlp(cleaned)
    sentiment = max(doc.cats, key=doc.cats.get)
    confidence = max(doc.cats.values())
    
    predictions.append({
        "Original Tweet": tweet,
        "Cleaned Text": cleaned,
        "Predicted Sentiment": sentiment,
        "Confidence": confidence,
        "Negative Score": doc.cats.get("negative", 0),
        "Positive Score": doc.cats.get("positive", 0),
        "Neutral Score": doc.cats.get("neutral", 0),
    })

# Create results DataFrame
results_df = pd.DataFrame(predictions)
print("\nBatch Prediction Results:")
print("="*100)
print(results_df.to_string(index=False))

# Summary statistics
print("\n" + "="*100)
print("Summary Statistics:")
print(results_df["Predicted Sentiment"].value_counts())
print(f"\nAverage Confidence: {results_df['Confidence'].mean():.4f}")



Batch Prediction Results:
                         Original Tweet                      Cleaned Text Predicted Sentiment  Confidence  Negative Score  Positive Score  Neutral Score
  I love this product! Amazing quality! love this product amazing quality            positive    0.460614        0.444288        0.460614       0.095098
This is absolutely terrible and useless  this absolutely terrible useless            positive    0.611873        0.367840        0.611873       0.020287
           The weather today is neutral             weather today neutral             neutral    0.822045        0.077495        0.100460       0.822045
           I hate waiting in long lines           hate waiting long lines            negative    0.757246        0.757246        0.116925       0.125829
      Best purchase I've made in years!          best purchase made years            positive    0.831596        0.067275        0.831596       0.101129
        Not bad, could be better though               c